# 🐍 Master Guide: Closures & Decorators
> **Single-Source Revision Guide**: Covers lexical closures (`__closure__` cell objects), function decorators, parameterized decorators, class decorators, and `functools.wraps`.
>
> *Run each cell interactively with `Shift + Enter`.*

---

## 📌 1. Topic Overview
A **Closure** is a function that retains access to variables from its enclosing lexical scope even after that scope has finished executing. **Decorators** are higher-order functions that dynamically modify or enhance another function's behavior without modifying its source code.

---
## 🧠 2. CPython Architecture: Cell Objects & Syntactic Sugar

### How Closures Work in CPython (`PyCellObject`)
When an inner function references a variable from an outer function, CPython creates a **cell object** on the heap (`PyCellObject`). Both functions reference this shared cell. When the outer function exits and its stack frame is destroyed, the cell object remains alive on the heap, referenced by `inner.__closure__`!

### Decorator Syntactic Sugar
```python
@my_decorator
def target(): pass
```
is 100% equivalent to:
```python
target = my_decorator(target)
```

---
## 🟢 3. Part 1: Beginner Fundamentals

### Step 1: Lexical Closures in Action

In [ ]:
def make_multiplier(factor):
    def multiply(x):
        return x * factor  # 'factor' is captured from outer scope
    return multiply

double = make_multiplier(2)
triple = make_multiplier(3)

print("double(5):", double(5))
print("triple(5):", triple(5))

# Inspecting the CPython closure cell object:
print("double.__closure__:", double.__closure__)
print("Captured cell value:", double.__closure__[0].cell_contents)

### Step 2: Creating a Standard Function Decorator
Always use `@functools.wraps(func)` to preserve the original function's `__name__` and `__doc__`!

In [ ]:
from functools import wraps
import time

def time_it(func):
    @wraps(func)  # Preserves name and metadata
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        result = func(*args, **kwargs)
        t1 = time.perf_counter()
        print(f"[{func.__name__}] Execution took {(t1 - t0)*1000:.3f} ms")
        return result
    return wrapper

@time_it
def compute_squares(n):
    """Calculates sum of squares."""
    return sum(x**2 for x in range(n))

print("Result:", compute_squares(50_000))
print("Preserved function name:", compute_squares.__name__)
print("Preserved docstring    :", compute_squares.__doc__)

---
## 🟡 4. Part 2: Intermediate Features

### Decorators That Accept Arguments (3 Levels of Nested Functions)
`@repeat(num_times=3)` requires an outer factory function that returns the actual decorator.

In [ ]:
def repeat(num_times):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(num_times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(num_times=3)
def greet(name):
    print(f"Hello, {name}!")

greet("Alice")

---
## 🔴 5. Part 3: Tricky Parts, Gotchas & Interview Traps

### ⚠️ Gotcha 1: Forgetting `@wraps` Metadata Obliteration
Without `@wraps(func)`, the decorated function loses its identity! Its name becomes `wrapper`, its docstring becomes `None`, and debugging tools break.

In [ ]:
def bad_decorator(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@bad_decorator
def special_function():
    """Important docstring."""
    pass

print("Function name ruined :", special_function.__name__)  # 'wrapper'!
print("Docstring lost       :", special_function.__doc__)   # None!
print("Rule: ALWAYS use @functools.wraps(func) on wrapper functions!")

---
## 📋 6. Part 4: Quick Revision Cheat Sheet

```
+------------------------------------+---------------------------------------+
| Concept                            | Key Rule                              |
+------------------------------------+---------------------------------------+
| Closure                            | Function + enclosing scope cell refs  |
| @decorator                         | func = decorator(func)                |
| @functools.wraps                   | Preserves __name__, __doc__, __module__|
| Parameterized decorator            | Requires 3-level function nesting     |
| *args, **kwargs in wrapper         | Allows decorator to wrap any signature|
+------------------------------------+---------------------------------------+
```

---
## 🎓 Practical Hands-On & Real-World Examples

### 1. The Decorator Mechanism (`DecEx1.py` - `DecEx3.py`)
> **Core Definition**: A decorator is a higher-order function that takes a base function as an argument,
> extends or wraps its execution with additional behavior, and returns a modified callable wrapper.

In [ ]:
# Practical Implementation: DecEx1.py - DecEx3.py
# 1. Output transformation decorator: Squaring numerical results
def square_decorator(func):
    def wrapper(*args, **kwargs):
        original_result = func(*args, **kwargs)
        transformed = original_result ** 2
        return transformed
    return wrapper

@square_decorator
def calculate_base_sum(a, b):
    return a + b

print(f"calculate_base_sum(3, 4) -> (3+4)^2 = {calculate_base_sum(3, 4)}")

# 2. String case modifier decorator (DecEx3.py)
def uppercase_decorator(func):
    def wrapper(*args, **kwargs):
        text = func(*args, **kwargs)
        return text.upper()
    return wrapper

@uppercase_decorator
def generate_greeting(name):
    return f"welcome to python technologies, {name}"

print("Decorated Greeting:", generate_greeting("Vikram"))

### 2. Execution Timer & Performance Profiling Decorator
A production-grade timing decorator measuring function execution latency.

In [ ]:
# Production Timer Decorator
import time
from functools import wraps

def timing_profiler(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start_time
        print(f"[PROFILER] '{func.__name__}' executed in {elapsed:.6f} seconds")
        return result
    return wrapper

@timing_profiler
def compute_sum_of_squares(n):
    return sum(i**2 for i in range(n))

res = compute_sum_of_squares(200_000)
print(f"Sum result: {res}")